# Sphota v2 decision run on a free Kaggle GPU

This notebook answers one question: **with a stronger multilingual model, can Sphota's fine codes pass the D-017/D-018 decision gate?**
It builds candidate spaces with three anchors (multilingual-e5-small, LaBSE, bge-m3), clusters the top level at 256 / 1024 / 4096 regions, and prints PASS/FAIL per criterion.

**Before you run** (right-hand panel → *Session options*):
1. **Accelerator:** GPU T4 ×2 or GPU P100.
2. **Internet:** on (models and data come from Hugging Face).

**Time:** about 2–4 hours at `medium` scale (12 languages, ~450k sentences). Kaggle sessions stop after 12 hours; the run resumes from its caches if you re-run.
**Cost:** ₹0 (Kaggle's free weekly GPU quota).

In [ ]:
# 1. Get the code (public repository: no token needed, and none should be added)
import os, subprocess
if not os.path.exists("/kaggle/working/sphota"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Thiruvikramanpravartak/sphota.git", "/kaggle/working/sphota"], check=True)
os.chdir("/kaggle/working/sphota")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Install (Kaggle already has CUDA torch; this adds the rest)
!pip -q install -e ".[build,encode]"
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on a GPU accelerator")

In [ ]:
# 3. Quick smoke test on tiny data (~5 min): proves the pipeline works on this machine
!bash scripts/gpu_build_v2.sh --scale tiny --dry-run 2>&1 | tail -12

In [ ]:
# 4. The decision run (~2-4 h). Caches live in data/ and .cache/; re-running resumes.
import os
os.environ["SPHOTA_THREADS"] = str(os.cpu_count())
!bash scripts/gpu_build_v2.sh --scale medium --sizes "256 1024 4096" 2>&1 | grep -E "==|D-017|PASS|FAIL|OVERALL|Traceback|Error" 

In [ ]:
# 5. Results: D-017 / D-018 PASS/FAIL per anchor and region count
import glob, json
GATE = {"1_coverage": ">= 60%", "2_translation_confident": ">= 80%", "3_cross_model": ">= 60%", "4_collapse_max": "<= 5%"}
for f in sorted(glob.glob("bench/results/v2-gate-medium-*.json")):
    r = json.load(open(f))
    print("\n" + f.split("/")[-1])
    for m in r["results"]:
        cells = " | ".join(f"{k.split('_',1)[1]} {100*v['value']:.1f}% {'PASS' if v['pass'] else 'FAIL'}"
                           for k, v in m["verdict"].items() if v["value"] is not None)
        allp = all(v["pass"] for v in m["verdict"].values()) and m["K1"] >= 256
        print(f"  K1={m['K1']:>4}: {cells}  ->  {'GATE PASSED' if allp else 'gate failed'}")

In [ ]:
# 6. Save the results to download (Output tab -> sphota-v2-results.zip)
!zip -qr /kaggle/working/sphota-v2-results.zip bench/results experiments -x "experiments/v2-tiny-*"
print("Download sphota-v2-results.zip from the Output tab, then add bench/results/*.json to the repo.")

## Reading the result
- **Every criterion PASS at ≥ 256 regions** for some anchor → the fine code can be marketed as a cross-language coordinate system; plan space `s2` (RFC + migration table).
- **Otherwise** → keep the current positioning: 32 regions for cross-language work, stored canonical vectors for model migration, fine codes as within-model precision.
Thresholds: `DECISIONS.md` D-017 and D-018. Guide: `docs/gpu-run.md`.